# Musanga V5 — NB08 within-landscape satellite detectability
CPU. Primary evaluation: 500 m spatial blocks, 100 m centre exclusion, all eligible training cells. No new LOFO runs. Historical results remain on Drive. Existing V5 S2, embedding and UAV caches are preserved. NICFI access is required for its supplementary branch.

## 00 — Native V5 source and cache configuration

In [ ]:
%pip -q install earthengine-api rasterio geopandas pyarrow scikit-learn scipy pandas matplotlib requests joblib
from pathlib import Path
from contextlib import ExitStack
from datetime import datetime,timezone
from collections import defaultdict
import os,json,hashlib,time,math,requests
import numpy as np,pandas as pd,geopandas as gpd,matplotlib.pyplot as plt
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.transform import from_origin,Affine
from rasterio.io import MemoryFile
from rasterio.warp import transform_bounds
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import roc_auc_score,average_precision_score
from scipy.stats import spearmanr
from threadpoolctl import threadpool_limits
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
threadpool_limits(limits=1)
V5_ROOT=next((p for p in (Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/V5'),Path('/content/drive/MyDrive/Musanga/Final/V5')) if (p.parent/'Data').is_dir() and p.is_dir()),None)
assert V5_ROOT is not None
ROOT=V5_ROOT/'Outputs/08_satellite_reference';ROOT.mkdir(parents=True,exist_ok=True)
OUT=V5_ROOT/'Outputs/08_satellite_science_v5';OUT.mkdir(parents=True,exist_ok=True)
V4_OUT=OUT # compatibility variable for retained analysis functions; never a V4 path
NB07=V5_ROOT/'Outputs/07_RGB_unified_production_v5'
MANIFEST=NB07/'07_production_mosaic_manifest.json'
assert MANIFEST.is_file(),'NB07 must finish all six mosaics before NB08 execution.'
M=json.loads(MANIFEST.read_text())
assert M['complete'] and M['schema']=='musanga_v5_nb07_rgb_mosaics' and len(M['mosaics'])==6
SITES=('Baego','MOS','Yangambi','Yoko')
YEARS={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2020}
KEYS=['site','mosaic','row','col']
CRS='EPSG:32635';TILE_M=1500;LATTICE_M=1;MIN_SUPPORT=.90
SCALES=[10,25,30,50,75,100]
EE_PROJECT='musanga-511110' # change to an authorized project when transferring
S2_SOURCE='COPERNICUS/S2_SR_HARMONIZED'
EMB_SOURCE='GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL'
S2_BANDS=['B2','B3','B4','B8','B5','B6','B7','B8A','B11','B12']
SCL_REJECT=[0,1,3,7,8,9,10,11]
INDEX_NAMES=['NDVI','NDMI','NBR','NDRE']
EMB_BANDS=[f'A{i:02d}' for i in range(64)]
ANNUAL=[f'RAW_ANNUAL__{i:02d}' for i in range(1,16)]
QUARTER=[f'S2_QUARTERLY_3YR__{i:02d}' for i in range(1,45)]
EMB=[f'EMB_ANNUAL__{i:02d}' for i in range(1,65)]
Q_COUNTS=[QUARTER[i-1] for i in (11,22,33,44)]
S2=ANNUAL+[c for c in QUARTER if c not in Q_COUNTS]
FEATURE_SETS={'M1_S2':S2,'M2_EMB':EMB,'M3_S2_EMB':S2+EMB}
ALL_FEATURES=sorted(set(sum(FEATURE_SETS.values(),[])))
SEED=20261120;SEED17=SEED;CAP_PER_TRAIN_FOREST=1500
MODEL=dict(learning_rate=.06,max_iter=180,max_leaf_nodes=20,min_samples_leaf=35,l2_regularization=2,early_stopping=False)
THRESHOLDS=(.05,.20,.40)
RUN_STRUCTURE=True  # supplementary; independent GEE assets may require access
RUN_ABUNDANCE_SENSITIVITY=True
def digest(obj):return hashlib.sha256(json.dumps(obj,sort_keys=True,default=str).encode()).hexdigest()
def atomic_json(p,obj):
 p=Path(p);tmp=p.with_suffix(p.suffix+'.partial');tmp.write_text(json.dumps(obj,indent=2,default=str));os.replace(tmp,p)
def atomic_parquet(d,p):
 p=Path(p);tmp=p.with_suffix('.partial');d.to_parquet(tmp,index=False);os.replace(tmp,p)
def dataframe_sha(d):return hashlib.sha256(pd.util.hash_pandas_object(d,index=False).values.tobytes()).hexdigest()
# CPU-only recovery of a missing final mosaic from its verified native V5 stripes.
# Never use a legacy raster, never rerun GPU inference, never overwrite an existing final.
import shutil
from rasterio.windows import Window
def recover_missing_mosaic(item,p):
 if p.is_file():return
 key=item['digest'];w=int(item['width']);h=int(item['height'])
 tr=Affine(*item['transform'][:6]);crs=rasterio.crs.CRS.from_string(item['crs'])
 stripe_h=1024
 folder=NB07/'stripe_cache'/item['name']/key[:20]
 expected=math.ceil(h/stripe_h)
 assert folder.is_dir(),f'Missing final mosaic AND stripe cache: {p}'
 stripes=[];total=0;tiles=0
 for i in range(expected):
  a=folder/f'{i:05d}.tif';meta=a.with_suffix('.json')
  assert a.is_file() and meta.is_file(),f'Missing stripe {i}: {folder}'
  rec=json.loads(meta.read_text())
  assert rec['signature']==key and rec['stripe']==i
  with rasterio.open(a) as ds:
   assert ds.tags().get('signature')==key and ds.tags().get('model_sha')==M['model_sha'] and ds.tags().get('policy_sha')==M['policy_sha']
   assert ds.count==2 and ds.dtypes==('float32','float32') and ds.nodata==-9999 and ds.crs==crs
   assert ds.width==w and ds.height==min(stripe_h,h-i*stripe_h)
   assert ds.transform.almost_equals(tr*Affine.translation(0,i*stripe_h))
  total+=int(rec['valid_pixels']);tiles+=int(rec['tiles']);stripes.append((a,rec))
 assert total==item['valid_pixels'] and tiles==item['count']
 print(f'RECOVER {item["name"]}: {expected} verified stripe headers; CPU assembly only',flush=True)
 local=Path('/content')/f'{item["name"]}_{key[:12]}_recovery.tif'
 profile=dict(driver='GTiff',width=w,height=h,count=2,dtype='float32',nodata=-9999,crs=crs,transform=tr,tiled=True,blockxsize=256,blockysize=256,compress='DEFLATE',predictor=3,BIGTIFF='YES',SPARSE_OK='TRUE')
 with rasterio.open(local,'w',**profile) as dst:
  dst.set_band_description(1,'Musanga probability');dst.set_band_description(2,'Musanga binary at 0.5')
  dst.update_tags(signature=key,model_sha=M['model_sha'],policy_sha=M['policy_sha'])
  for i,(a,rec) in enumerate(stripes):
   count=0
   with rasterio.open(a) as src:
    for _,win in src.block_windows(1):
     z=src.read(window=win);v=z[0]!=-9999
     assert np.array_equal(v,z[1]!=-9999)
     assert np.isfinite(z).all() and np.all((z[0][v]>=0)&(z[0][v]<=1))
     assert np.array_equal(z[1][v],(z[0][v]>=.5).astype(np.float32))
     count+=int(v.sum())
     if v.any():dst.write(z,window=Window(win.col_off,win.row_off+i*stripe_h,win.width,win.height))
   assert count==rec['valid_pixels'],(a,count,rec['valid_pixels'])
   print(f'Recovery {item["name"]}: stripe {i+1}/{expected}',flush=True)
 # Copy to final name directly (avoid the previous Drive rename failure).
 # A completion receipt is written only after read-back byte hash verification.
 assert not p.exists(),f'Final appeared during recovery: {p}; stop concurrent writers'
 def sha_file(q):
  h=hashlib.sha256()
  with open(q,'rb') as f:
   for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
  return h.hexdigest()
 digest_local=sha_file(local)
 with open(local,'rb') as f,open(p,'xb') as out:
  shutil.copyfileobj(f,out,8*1024*1024);out.flush();os.fsync(out.fileno())
 assert p.stat().st_size==local.stat().st_size and sha_file(p)==digest_local
 with rasterio.open(p) as ds:
  assert ds.width==w and ds.height==h and ds.tags().get('signature')==key
 atomic_json(p.with_suffix('.recovery.json'),dict(signature=key,sha256=digest_local,valid_pixels=total,stripes=expected))
 local.unlink()
 print('RECOVERED AND VERIFIED',p,flush=True)

SOURCE_META=[];MOSAICS=defaultdict(list)
for item in M['mosaics']:
 p=NB07/'reconstructed_mosaics'/Path(item['output']).name
 rec=json.loads((p.parent/(item['name']+'_reconstruction.json')).read_text())
 assert rec['status']=='complete' and rec['digest']==item['digest'] and rec['model']==M['model_sha'] and rec['policy']==M['policy_sha']
 recover_missing_mosaic(item,p)
 with rasterio.open(p) as ds:
  assert ds.count==2 and ds.nodata==-9999 and ds.crs==rasterio.crs.CRS.from_string(CRS)
  assert ds.tags().get('signature')==item['digest']
  assert ds.width==item['width'] and ds.height==item['height']
  assert ds.descriptions==('Musanga probability','Musanga binary at 0.5'),ds.descriptions
  bounds=list(ds.bounds)
 MOSAICS[item['site']].append((item['name'],p,bounds))
 SOURCE_META.append(dict(name=item['name'],site=item['site'],digest=item['digest'],size=p.stat().st_size,mtime_ns=p.stat().st_mtime_ns,bounds=bounds))
assert set(MOSAICS)==set(SITES)
for k in MOSAICS:MOSAICS[k].sort(key=lambda t:t[0])
TARGET_CFG=dict(version='v5_native_grid_v1',model=M['model_sha'],policy=M['policy_sha'],mosaics=SOURCE_META,crs=CRS,origin=[0,0],tile_m=TILE_M,lattice_m=LATTICE_M,scales=SCALES,support=MIN_SUPPORT,method='1m nearest valid support and binary point sampling; best-supported mosaic per cell; no blending')
TARGET_ID=digest(TARGET_CFG)[:20];TARGET_DIR=ROOT/'uav_targets'/TARGET_ID;TARGET_DIR.mkdir(parents=True,exist_ok=True)
atomic_json(TARGET_DIR/'config.json',TARGET_CFG)
# Geometry is only used for candidate tile pruning; the map's actual NoData mask is authoritative.
raw=json.loads((V5_ROOT/'Outputs/00_input_audit/00_input_manifest.json').read_text())
roi_path=Path(raw['roi'])
if not roi_path.is_file():roi_path=V5_ROOT.parent/'Data/Ortho_RGB'/roi_path.name
roi=unary_union(gpd.read_file(roi_path).to_crs(CRS).geometry).buffer(-100.)
assert roi.is_valid and not roi.is_empty
TILES={}
for forest in SITES:
 keys=set()
 for name,p,b in MOSAICS[forest]:
  geom=roi.intersection(box(*b));prepared=prep(geom)
  if geom.is_empty:continue
  xmin,ymin,xmax,ymax=geom.bounds
  for tx in range(math.floor(xmin/TILE_M),math.ceil(xmax/TILE_M)):
   for ty in range(math.floor(ymin/TILE_M),math.ceil(ymax/TILE_M)):
    if prepared.intersects(box(tx*TILE_M,ty*TILE_M,(tx+1)*TILE_M,(ty+1)*TILE_M)):keys.add((tx,ty))
 TILES[forest]=sorted(keys)
 assert keys,forest
print('Candidate tiles per forest:',{k:len(v) for k,v in TILES.items()})
print('Native V5 target fingerprint:',TARGET_ID)


## 01 — Visual inspection of RGB, annotations and NB07 full-data maps

In [ ]:
# Production-map sanity check, NOT independent validation (NB07 trained on these annotations).
from rasterio.features import rasterize
from shapely import make_valid
from matplotlib.colors import ListedColormap
AOUT=OUT/'visual_UAV_QA'/TARGET_ID/'three_examples_probability_innerROI';AOUT.mkdir(parents=True,exist_ok=True)
base=V5_ROOT/'Outputs'
units=gpd.read_file(base/'01_spatial_cover_folds_v2/01_reference_groups_and_folds_v2.gpkg').to_crs(CRS)
units['unit_id']=units.unit_id.astype(str)
owners=pd.read_csv(base/'02_rgb_circles_plus_plots_v3/02_reference_mosaic_owners_v2.csv',dtype={'unit_id':str})
scores=pd.read_csv(base/'04_unified_rgb_heldout_v5/04_heldout_unit_metrics.csv',dtype={'unit_id':str})
units=units.merge(owners[['unit_type','unit_id','mosaic','roi_mosaic_coverage_fraction']],on=['unit_type','unit_id'],validate='one_to_one')
units=units.merge(scores[['unit_type','unit_id','observed_cover_pp','abs_cover_error_pp']],on=['unit_type','unit_id'],validate='one_to_one')
f78=json.loads((base/'00_input_audit/00_v4_field78_contract.json').read_text())
def readgeo(p):
 p=Path(p)
 if not p.exists():raise FileNotFoundError(p)
 g=gpd.read_file(p).to_crs(CRS)
 g.geometry=g.geometry.map(make_valid)
 return g
circles=readgeo(f78['circle_positive_crowns'])
crowns=readgeo(f78['exhaustive_plot_crowns'])
crowns=crowns[pd.to_numeric(crowns.Class_ID,errors='raise').eq(1)]
rgb_paths={Path(d['file']).stem:Path(d['path']) for d in raw['mosaics']}
pred_paths={name:p for forest in SITES for name,p,b in MOSAICS[forest]}
# Choose three distinct forests, with annotation cover closest to 5%, 30%, 65%.
# Prefer circles; use exhaustive plots only if no feasible three-forest circle set.
from itertools import permutations
# Require the entire displayed window (including 5 m context and pixel rounding)
# to be within the 100 m inner ROI and the owning production mosaic bounds.
mosaic_bounds={name:box(*b) for forest in SITES for name,p,b in MOSAICS[forest]}
def interior_example(u):
 geom=make_valid(u.geometry)
 view=box(*geom.buffer(6).bounds)
 name=Path(u.mosaic).stem
 return name in mosaic_bounds and roi.covers(view) and mosaic_bounds[name].covers(view)
eligible=units[units.roi_mosaic_coverage_fraction.ge(.99)].copy()
eligible=eligible[eligible.apply(interior_example,axis=1)].copy()
pool=eligible[eligible.unit_type.eq('circle')].copy()
if pool.site.nunique()<3:pool=eligible.copy()
print('Unclipped candidates by forest:',pool.groupby('site').size().to_dict())
assert pool.site.nunique()>=3,'Fewer than three forests with fully interior examples; do not relax the ROI buffer.'
targets=[5.,30.,65.]
choices=[]
for forests in permutations(sorted(pool.site.unique()),3):
 ids=[]
 for forest,target in zip(forests,targets):
  g=pool[pool.site.eq(forest)]
  ids.append((g.observed_cover_pp-target).abs().idxmin())
 cost=sum(abs(float(pool.loc[ix,'observed_cover_pp'])-target) for ix,target in zip(ids,targets))
 choices.append((cost,ids))
assert choices
ids=min(choices,key=lambda t:t[0])[1]
display(pool.loc[ids,['site','unit_type','unit_id','observed_cover_pp']])
qa_rows=[]
fig,axs=plt.subplots(3,5,figsize=(16,9),squeeze=False,layout='constrained')
for row,ix in enumerate(ids):
  u=pool.loc[ix];forest=u.site;geom=make_valid(u.geometry);name=Path(u.mosaic).stem
  xmin,ymin,xmax,ymax=geom.buffer(5).bounds
  step=max(.20,(xmax-xmin)/500,(ymax-ymin)/500)
  w=math.ceil((xmax-xmin)/step);h=math.ceil((ymax-ymin)/step)
  tf=from_origin(xmin,ymax,step,step);extent=(xmin,xmin+w*step,ymax-h*step,ymax)
  with rasterio.open(rgb_paths[name]) as ds, WarpedVRT(ds,crs=CRS,transform=tf,width=w,height=h,resampling=Resampling.nearest) as vrt:
   rgb=vrt.read([1,2,3],masked=True).filled(0).transpose(1,2,0)
  with rasterio.open(pred_paths[name]) as ds,WarpedVRT(ds,crs=CRS,transform=tf,width=w,height=h,resampling=Resampling.nearest,nodata=-9999) as vrt:
   p=vrt.read(1,masked=True).filled(-9999)
  truth=circles if u.unit_type=='circle' else crowns
  sub=truth[truth.intersects(geom)]
  shapes=[(a.intersection(geom),1) for a in sub.geometry if not a.intersection(geom).is_empty]
  mask=rasterize([(geom,1)],out_shape=(h,w),transform=tf,dtype='uint8').astype(bool)
  ann=rasterize(shapes,out_shape=(h,w),transform=tf,dtype='uint8').astype(bool) if shapes else np.zeros((h,w),bool)
  valid=mask&np.isfinite(p)&(p>=0)&(p<=1)
  pred=p>=.5
  def overlay(a,m,color):
   rgba=np.zeros((*m.shape,4),dtype=np.float32);rgba[:,:,:3]=color;rgba[:,:,3]=m*.48
   a.imshow(rgba,extent=extent,interpolation='nearest')
  # Same geographic extent in all panels; white outlines are annotated crowns.
  for j,ax in enumerate(axs[row]):
   if j in (0,1,4):ax.imshow(rgb,extent=extent)
   elif j==2:ax.imshow(np.ma.masked_where((p<0)|(p>1),p),extent=extent,cmap='magma',vmin=0,vmax=1,interpolation='nearest')
   else:ax.imshow(np.ma.masked_where((p<0)|(p>1),pred.astype(float)),extent=extent,cmap=ListedColormap(['#f5f5ed','#004529']),vmin=0,vmax=1,interpolation='nearest')
   if j!=0:
    gpd.GeoSeries([geom],crs=CRS).boundary.plot(ax=ax,color='white',linewidth=.8)
    if shapes:gpd.GeoSeries([g for g,v in shapes],crs=CRS).boundary.plot(ax=ax,color='white',linewidth=.65)
   ax.set_xlim(extent[:2]);ax.set_ylim(extent[2:]);ax.set_axis_off()
  overlay(axs[row,4],pred&valid,[1,.15,.1])
  obs=100*ann[valid].mean() if valid.any() else np.nan
  est=100*pred[valid].mean() if valid.any() else np.nan
  axs[row,0].set_title(f'{forest} {u.unit_type} {u.unit_id} | RGB')
  axs[row,1].set_title(f'Annotations | {obs:.1f}%')
  axs[row,2].set_title('Probability (0–1)')
  axs[row,3].set_title(f'Binary Musanga | {est:.1f}%')
  axs[row,4].set_title(f'Prediction overlay | valid {valid.sum()/max(1,mask.sum()):.0%}')
  qa_rows.append(dict(site=forest,unit_type=u.unit_type,unit_id=u.unit_id,mosaic=name,display_pixel_m=step,observed_pp=obs,predicted_pp=est,valid_fraction=valid.sum()/max(1,mask.sum())))
fig.suptitle('Production map visual QA — training footprints; display-grid approximation, not held-out scores')
fig.savefig(AOUT/'three_annotation_RGB_prediction.png',dpi=130);plt.show();plt.close(fig)
display(pd.DataFrame(qa_rows));pd.DataFrame(qa_rows).to_csv(AOUT/'visual_QA.csv',index=False)


## 02 — Cached UAV fractional cover and area QA

In [ ]:
# One bounded 1m lattice per tile and mosaic; all six scales derived from that lattice.
# A nearest-neighbour lattice is an area approximation, not exact native-pixel area.
def tile_bounds(tx,ty):return (tx*TILE_M,ty*TILE_M,(tx+1)*TILE_M,(ty+1)*TILE_M)
def tile_transform(tx,ty,spacing):
 x0,y0,x1,y1=tile_bounds(tx,ty);return from_origin(x0,y1,spacing,spacing)
def reduce_grid(a,scale,spacing):
 n=int(scale/spacing);h,w=a.shape;assert h%n==0 and w%n==0
 return a.reshape(h//n,n,w//n,n).sum(axis=(1,3))
TARGET_COLS=['site','mosaic','scale_m','row','col','x','y','target_fraction','mean_probability','uav_valid_fraction','tile_x','tile_y']
def target_tile(forest,tx,ty,opened):
 dest=TARGET_DIR/f'{forest}_{tx}_{ty}.parquet'
 if dest.is_file():
  d=pd.read_parquet(dest);assert list(d.columns)==TARGET_COLS
  return d
 b=tile_bounds(tx,ty);tr=tile_transform(tx,ty,LATTICE_M);n=TILE_M//LATTICE_M
 best={}
 for scale in SCALES:
  k=TILE_M//scale;best[scale]=(np.zeros((k,k),np.float32),np.full((k,k),np.nan,np.float32),np.full((k,k),np.nan,np.float32),np.full((k,k),-1,np.int16))
 for j,(name,p,bounds) in enumerate(MOSAICS[forest]):
  if not box(*b).intersects(box(*bounds)):continue
  ds=opened[name]
  with WarpedVRT(ds,crs=CRS,transform=tr,width=n,height=n,resampling=Resampling.nearest,nodata=-9999.) as vrt:
   z=vrt.read([1,2],masked=True)
  a=z.filled(-9999.).astype(np.float32,copy=False)
  valid=(~np.ma.getmaskarray(z).any(axis=0))&np.isfinite(a).all(axis=0)&(a[0]>=0)&(a[0]<=1)&((a[1]==0)|(a[1]==1))
  assert np.array_equal(a[1][valid],(a[0][valid]>=.5).astype(np.float32))
  if not valid.any():continue
  for scale in SCALES:
   count=reduce_grid(valid.astype(np.int32),scale,LATTICE_M)
   support=count/float((scale/LATTICE_M)**2)
   total=reduce_grid(np.where(valid,a[1],0.),scale,LATTICE_M)
   prob=reduce_grid(np.where(valid,a[0],0.),scale,LATTICE_M)
   v,c,pb,idx=best[scale];take=(support>=MIN_SUPPORT)&(support>v)
   v[take]=support[take];c[take]=total[take]/count[take];pb[take]=prob[take]/count[take];idx[take]=j
 frames=[]
 for scale,(v,c,pb,idx) in best.items():
  rr,cc=np.where(idx>=0)
  if not len(rr):continue
  x=b[0]+(cc+.5)*scale;y=b[3]-(rr+.5)*scale
  frames.append(pd.DataFrame(dict(site=forest,mosaic=[MOSAICS[forest][j][0] for j in idx[rr,cc]],scale_m=scale,
   row=np.floor(-y/scale).astype('int64'),col=np.floor(x/scale).astype('int64'),x=x,y=y,target_fraction=c[rr,cc],mean_probability=pb[rr,cc],uav_valid_fraction=v[rr,cc],tile_x=tx,tile_y=ty)))
 d=pd.concat(frames,ignore_index=True)[TARGET_COLS] if frames else pd.DataFrame(columns=TARGET_COLS)
 assert not d.duplicated(['site','scale_m','row','col']).any()
 atomic_parquet(d,dest)
 return d
# Fast path: load the completed aggregate for this exact source/recipe fingerprint.
# The historical aggregate was atomically saved after processing every candidate tile.
TARGET_AGG=TARGET_DIR/'all_scales_uav_reference.parquet'
FORCE_TARGET_REBUILD=False  # True only when intentionally auditing/rebuilding the aggregate
expected_tiles={(f,int(tx),int(ty)) for f in SITES for tx,ty in TILES[f]}
def validate_targets(d):
 assert list(d.columns)==TARGET_COLS
 assert not d.duplicated(['site','scale_m','row','col']).any()
 assert d.target_fraction.between(0,1).all() and d.mean_probability.between(0,1).all()
 assert d.uav_valid_fraction.between(MIN_SUPPORT-1e-6,1+1e-6).all()
 assert set(d.scale_m)==set(SCALES) and d.groupby('scale_m').site.nunique().eq(len(SITES)).all()
 assert set(d[['site','tile_x','tile_y']].drop_duplicates().itertuples(index=False,name=None)).issubset(expected_tiles)
 assert all((TARGET_DIR/f'{f}_{x}_{y}.parquet').is_file() for f,x,y in expected_tiles)
if TARGET_AGG.is_file() and not FORCE_TARGET_REBUILD:
 TARGET=pd.read_parquet(TARGET_AGG)
 validate_targets(TARGET)
 print(f'UAV AGGREGATE CACHE: {len(TARGET):,} cells | {TARGET_AGG}',flush=True)
else:
 target_parts=[];t0=time.monotonic()
 for forest in SITES:
  with ExitStack() as stack:
   opened={name:stack.enter_context(rasterio.open(p)) for name,p,b in MOSAICS[forest]}
   for i,(tx,ty) in enumerate(TILES[forest],1):
    d=target_tile(forest,tx,ty,opened)
    if len(d):target_parts.append(d)
    print(f'UAV {forest} {i}/{len(TILES[forest])}: {len(d):,} cells across scales | {(time.monotonic()-t0)/60:.1f} min',flush=True)
 TARGET=pd.concat(target_parts,ignore_index=True)
 del target_parts
 validate_targets(TARGET)
 atomic_parquet(TARGET,TARGET_AGG)
assert not TARGET.duplicated(['site','scale_m','row','col']).any()
assert TARGET.target_fraction.between(0,1).all() and TARGET.uav_valid_fraction.ge(MIN_SUPPORT-1e-6).all()
assert TARGET.groupby('scale_m').site.nunique().eq(4).all()
display(TARGET.groupby(['scale_m','site']).target_fraction.agg(n='size',mean='mean',max='max').round(3))


In [ ]:
# Audit the approximation against exact native-pixel overlap area on a fixed sample.
# This reads at most 24 individual 50m windows, not entire mosaics.
from rasterio.windows import from_bounds,Window
AREA_QA_PATH=TARGET_DIR/'01_native_area_spotcheck_50m.csv'
samples=[]
for forest,g in TARGET[TARGET.scale_m.eq(50)].groupby('site'):
 boundary=g[g.uav_valid_fraction.lt(.98)]
 interior=g[g.uav_valid_fraction.ge(.98)]
 for label,z in [('boundary',boundary),('interior',interior)]:
  if len(z):
   q=z.sample(n=min(3,len(z)),random_state=SEED).copy();q['sample_type']=label;samples.append(q)
sample=pd.concat(samples,ignore_index=True)
if not AREA_QA_PATH.is_file():
 qa=[]
 for (forest,mosaic),g in sample.groupby(['site','mosaic']):
  p=next(p for name,p,b in MOSAICS[forest] if name==mosaic)
  with rasterio.open(p) as ds:
   tf=ds.transform;assert tf.a>0 and tf.e<0 and abs(tf.b)<1e-9 and abs(tf.d)<1e-9
   for row in g.itertuples():
    xmin=row.x-25;xmax=row.x+25;ymin=row.y-25;ymax=row.y+25
    window_raw=from_bounds(xmin,ymin,xmax,ymax,tf)
    c0=max(0,math.floor(window_raw.col_off));r0=max(0,math.floor(window_raw.row_off))
    c1=min(ds.width,math.ceil(window_raw.col_off+window_raw.width));r1=min(ds.height,math.ceil(window_raw.row_off+window_raw.height))
    assert c1>c0 and r1>r0
    a=ds.read(2,window=Window(c0,r0,c1-c0,r1-r0),masked=True)
    v=(~np.ma.getmaskarray(a))&np.isin(a.filled(-9999),[0,1])
    left=tf.c+np.arange(c0,c1)*tf.a
    top=tf.f+np.arange(r0,r1)*tf.e
    dx=np.maximum(0,np.minimum(left+tf.a,xmax)-np.maximum(left,xmin))
    dy=np.maximum(0,np.minimum(top,ymax)-np.maximum(top+tf.e,ymin))
    weights=dy[:,None]*dx[None,:]
    area=float(np.sum(weights*v));cover=float(np.sum(weights*np.where(v,a.filled(0),0))/area) if area>0 else np.nan
    qa.append(dict(site=forest,mosaic=mosaic,row=row.row,col=row.col,sample_type=row.sample_type,
      lattice_cover=row.target_fraction,native_cover=cover,lattice_support=row.uav_valid_fraction,native_support=area/2500,
      cover_delta_pp=100*(row.target_fraction-cover),support_delta_pp=100*(row.uav_valid_fraction-area/2500)))
 QAREA=pd.DataFrame(qa);tmp=AREA_QA_PATH.with_suffix('.partial');QAREA.to_csv(tmp,index=False);os.replace(tmp,AREA_QA_PATH)
else:QAREA=pd.read_csv(AREA_QA_PATH)
assert len(QAREA)==len(sample)
display(QAREA.round(4))
print('Native area spot-check | cover MAE (pp):',QAREA.cover_delta_pp.abs().mean(),'| max abs support error (pp):',QAREA.support_delta_pp.abs().max())
if QAREA.cover_delta_pp.abs().max()>2 or QAREA.support_delta_pp.abs().max()>2:
 print('WARNING: >2 pp spot-check difference; inspect lattice accuracy before publication.')


## 03 — Cached S2/embedding data and reference joins

In [ ]:
# Direct snapped 10 m Earth Engine downloads; no old predictor files.
import ee
try:ee.Initialize(project=EE_PROJECT)
except Exception:
 ee.Authenticate()
 ee.Initialize(project=EE_PROJECT)
SAT_CFG=dict(version='v5_ee_10m_v1',crs=CRS,tile_m=TILE_M,years=YEARS,s2=S2_SOURCE,embedding=EMB_SOURCE,mask=SCL_REJECT,annual_edge_mask=['B8A','B9'],annual_20m_resampling='bilinear',annual_indices=INDEX_NAMES,quarter_mask='SCL only, original NB04A',quarter_years='year-2 through year',reflectance_order=S2_BANDS,grid_origin=[0,0],scaling=.0001,brdf='none')
SAT_ID=digest(SAT_CFG)[:20];SAT_DIR=ROOT/'satellite_10m'/SAT_ID;SAT_DIR.mkdir(parents=True,exist_ok=True)
atomic_json(SAT_DIR/'config.json',SAT_CFG)
def ee_region(tx,ty):
 b=transform_bounds(CRS,'EPSG:4326',*tile_bounds(tx,ty),densify_pts=21)
 return ee.Geometry.Rectangle(list(b),proj='EPSG:4326',geodesic=False)
def clean_annual(im):
 valid=im.select('SCL').remap(SCL_REJECT,[0]*len(SCL_REJECT),1).eq(1)
 valid=valid.And(im.select('B8A').mask()).And(im.select('B9').mask())
 return ee.Image.cat([im.select(S2_BANDS[:4]),im.select(S2_BANDS[4:]).resample('bilinear')]).updateMask(valid).copyProperties(im,['system:time_start'])
def clean_quarter(im):
 valid=im.select('SCL').remap(SCL_REJECT,[0]*len(SCL_REJECT),1).eq(1)
 return im.select(S2_BANDS).updateMask(valid).multiply(.0001).copyProperties(im,['system:time_start'])
def indices(im):
 return ee.Image.cat([im,im.normalizedDifference(['B8','B4']).rename('NDVI'),im.normalizedDifference(['B8','B11']).rename('NDMI'),im.normalizedDifference(['B8','B12']).rename('NBR'),im.normalizedDifference(['B8A','B5']).rename('NDRE')])
def sat_image(forest,tx,ty,family):
 year=YEARS[forest];area=ee_region(tx,ty)
 if family=='EMB':
  coll=ee.ImageCollection(EMB_SOURCE).filterBounds(area).filterDate(f'{year}-01-01',f'{year+1}-01-01').sort('system:index')
  image=coll.mosaic().select(EMB_BANDS)
  bands=EMB_BANDS
 elif family=='ANNUAL':
  coll=ee.ImageCollection(S2_SOURCE).filterBounds(area).filterDate(f'{year}-01-01',f'{year+1}-01-01').map(clean_annual)
  image=ee.Image.cat([indices(coll.median().multiply(.0001)),coll.select('B2').count().unmask(0).rename('N_CLEAR')])
  bands=S2_BANDS+INDEX_NAMES+['N_CLEAR']
 else:
  q=int(family[1:]);coll=ee.ImageCollection(S2_SOURCE).filterBounds(area).filterDate(f'{year-2}-01-01',f'{year+1}-01-01').filter(ee.Filter.calendarRange(q*3-2,q*3,'month')).map(clean_quarter)
  image=ee.Image.cat([coll.median().select(S2_BANDS),coll.select('B2').count().unmask(0).rename('N_CLEAR')])
  bands=S2_BANDS+['N_CLEAR']
 return image.select(bands).unmask(-9999,sameFootprint=False).toFloat(),bands,coll
def fetch_image(image,bands,folder,forest,tx,ty,family,recipe_id,collection=None):
 folder.mkdir(parents=True,exist_ok=True)
 path=folder/f'{forest}_{tx}_{ty}_{family}.tif';meta=path.with_suffix('.json')
 tf=tile_transform(tx,ty,10);n=TILE_M//10
 def verify(p):
  with rasterio.open(p) as ds:
   assert ds.crs==rasterio.crs.CRS.from_string(CRS) and ds.shape==(n,n) and ds.count==len(bands)
   assert ds.transform.almost_equals(tf,precision=1e-6) and ds.nodata==-9999
   assert list(ds.descriptions)==bands and ds.tags().get('recipe')==recipe_id
   a=ds.read();assert np.isfinite(a).all()
   return a
 if path.is_file() and meta.is_file():
  assert json.loads(meta.read_text())['recipe']==recipe_id
  return verify(path)
 # Preserve exact catalogue image IDs in the V5 cache provenance.
 ids=collection.aggregate_array('system:index').getInfo() if collection is not None else None
 if collection is not None:assert len(ids)>0,(forest,tx,ty,family,'no scenes')
 params=dict(crs=CRS,crs_transform=list(tf)[:6],dimensions=[n,n],format='GEO_TIFF',filePerBand=False)
 error=None
 for attempt in range(4):
  try:
   url=image.getDownloadURL(params)
   response=requests.get(url,timeout=(30,300));response.raise_for_status()
   with MemoryFile(response.content) as mem,mem.open() as ds:
    assert ds.count==len(bands) and ds.shape==(n,n) and ds.crs==rasterio.crs.CRS.from_string(CRS),(family,ds.shape,ds.crs)
    assert ds.transform.almost_equals(tf,precision=1e-6),(family,ds.transform,tf)
    a=ds.read().astype(np.float32)
   assert np.isfinite(a).all()
   tmp=path.with_suffix('.partial.tif')
   with rasterio.open(tmp,'w',driver='GTiff',width=n,height=n,count=len(bands),dtype='float32',crs=CRS,transform=tf,nodata=-9999.,compress='deflate',tiled=True) as dst:
    dst.write(a)
    for j,b in enumerate(bands,1):dst.set_band_description(j,b)
    dst.update_tags(recipe=recipe_id,site=forest,family=family)
   os.replace(tmp,path);atomic_json(meta,dict(recipe=recipe_id,source_image_ids=ids,retrieved_utc=datetime.now(timezone.utc).isoformat(),bands=bands))
   return verify(path)
  except Exception as exc:
   error=exc;print('EE retry',forest,family,tx,ty,attempt+1,str(exc)[:220],flush=True);time.sleep(2**attempt)
 raise RuntimeError(f'EE download failed for {forest} {family} {tx} {ty}; completed caches are preserved') from error
def get_sat(forest,tx,ty,family):
 img,bands,coll=sat_image(forest,tx,ty,family)
 return fetch_image(img,bands,SAT_DIR,forest,tx,ty,family,SAT_ID,coll)
def aggregate_features(a,scale,count_indices=()):
 # Expand 10m pixels into equal 5m quadrants for exact aligned area weights
 # at 25/75m as well as integer multiples of 10m. No artificial new information.
 good=np.isfinite(a).all(axis=0)&(a!=-9999).all(axis=0)
 for i in count_indices:good &= a[i]>0
 base=np.repeat(np.repeat(good,2,axis=0),2,axis=1)
 n=scale//5;den=reduce_grid(base.astype(np.int32),scale,5)
 vals=[]
 for b in a:
  z=np.repeat(np.repeat(np.where(good,b,0.),2,axis=0),2,axis=1)
  sums=reduce_grid(z,scale,5)
  vals.append(np.divide(sums,den,out=np.full(sums.shape,np.nan,dtype=np.float32),where=den>0))
 return np.stack(vals),den/float(n*n)
JOIN_CFG=dict(version='v5_native_join_v1',target=TARGET_ID,satellite=SAT_ID,support=MIN_SUPPORT,feature_policy='90% jointly valid 10m samples within each family; common valid cells across families for 50m')
JOIN_ID=digest(JOIN_CFG)[:20];JOIN_DIR=ROOT/'joined_tiles'/JOIN_ID;JOIN_DIR.mkdir(parents=True,exist_ok=True)
atomic_json(JOIN_DIR/'config.json',JOIN_CFG)
def extract_join(forest,tx,ty,d):
 annual=get_sat(forest,tx,ty,'ANNUAL')
 extra=[]
 if d.scale_m.eq(50).any():
  extra=[(get_sat(forest,tx,ty,f'Q{q}'),[10],QUARTER[(q-1)*11:q*11]) for q in range(1,5)]
  extra.append((get_sat(forest,tx,ty,'EMB'),[],EMB))
 pieces=[]
 for scale,g in d.groupby('scale_m'):
  b=tile_bounds(tx,ty);rr=np.floor((b[3]-g.y.to_numpy())/scale).astype(int);cc=np.floor((g.x.to_numpy()-b[0])/scale).astype(int)
  assert np.all(rr>=0) and np.all(cc>=0) and np.all(rr<TILE_M//scale) and np.all(cc<TILE_M//scale)
  vals,cov=aggregate_features(annual,int(scale),[14])
  h=g.copy().reset_index(drop=True)
  for i,c in enumerate(ANNUAL):h[c]=vals[i,rr,cc]
  h['annual_support']=cov[rr,cc];h=h[h.annual_support.ge(MIN_SUPPORT-1e-6)].copy()
  h['all_features_valid']=False
  if scale==50 and len(h):
   rr=np.floor((b[3]-h.y.to_numpy())/scale).astype(int);cc=np.floor((h.x.to_numpy()-b[0])/scale).astype(int)
   support=np.ones(len(h),bool)
   for a,counts,names in extra:
    v,cov=aggregate_features(a,50,counts)
    h=pd.concat([h,pd.DataFrame(v[:,rr,cc].T,columns=names,index=h.index)],axis=1)
    support &= cov[rr,cc]>=MIN_SUPPORT-1e-6
   h['all_features_valid']=support
  pieces.append(h)
 return pd.concat(pieces,ignore_index=True)
# Keep wide 119-feature rows only at 50m; narrow tables at all other scales.
# Avoid concatenating millions of finer-scale rows with unused quarterly/embedding columns.
parts_by_scale=defaultdict(list)
groups=TARGET.groupby(['site','tile_x','tile_y'],sort=True);total=groups.ngroups
for i,((forest,tx,ty),d) in enumerate(groups,1):
 tx=int(tx);ty=int(ty);p=JOIN_DIR/f'{forest}_{tx}_{ty}.parquet'
 if p.is_file():
  z=pd.read_parquet(p);print('JOIN CACHE',forest,tx,ty,flush=True)
 else:
  z=extract_join(forest,tx,ty,d);atomic_parquet(z,p)
  print('JOIN SAVED',forest,tx,ty,len(z),flush=True)
 for k,q in z.groupby('scale_m'):
  cols=TARGET_COLS+ANNUAL+['annual_support']
  if k==50:cols+=QUARTER+EMB+['all_features_valid']
  # Some tiles lack 50m valid coverage; those never enter this branch.
  parts_by_scale[int(k)].append(q.reindex(columns=cols).copy())
 del z
 print(f'{i}/{total} satellite tiles complete',flush=True)
del groups
J50=pd.concat(parts_by_scale[50],ignore_index=True)
assert not J50.duplicated(['site','row','col']).any()
D=J50[J50.all_features_valid.eq(True)].copy()
D=D[np.isfinite(D[ALL_FEATURES+Q_COUNTS].to_numpy(float)).all(axis=1)&(D[Q_COUNTS].to_numpy(float)>=1).all(axis=1)].sort_values(KEYS).reset_index(drop=True)
assert set(D.site)==set(SITES) and not D.duplicated(['site','row','col']).any() and D.target_fraction.between(0,1).all()
D4=D.copy() # compatibility alias for retained scientific analysis code
SOURCE_SHA=dataframe_sha(D[KEYS+['x','y','target_fraction']+ALL_FEATURES])
CONFIG=dict(version='NB08_V5_native_50m',target_id=TARGET_ID,satellite_id=SAT_ID,join_id=JOIN_ID,source_sha256=SOURCE_SHA,n=len(D),features=FEATURE_SETS,seed=SEED,cap=CAP_PER_TRAIN_FOREST,model=MODEL,thresholds=THRESHOLDS)
SIGNATURE=digest(CONFIG)[:20];RUN=OUT/SIGNATURE;RUN.mkdir(parents=True,exist_ok=True)
atomic_json(RUN/'config.json',CONFIG)
REF=ROOT/f'native_50m_reference_{SIGNATURE}.parquet';atomic_parquet(D,REF)
scale_hashes={}
scale_cols=['site','scale_m','row','col','x','y','target_fraction','uav_valid_fraction']+ANNUAL[:14]
for k in SCALES:
 q=pd.concat([a[scale_cols] for a in parts_by_scale[k]],ignore_index=True).sort_values(['site','row','col']).reset_index(drop=True)
 assert not q.duplicated(['site','row','col']).any()
 assert set(q.site)==set(SITES) and np.isfinite(q[ANNUAL[:14]].to_numpy(float)).all()
 scale_hashes[str(k)]=dataframe_sha(q)
 del q
SCALE_CFG=dict(version='v5_annual_scale_v1',join=JOIN_ID,table_sha=scale_hashes,scales=SCALES)
SCALE_RUN=OUT/'native_scale_LOFO'/digest(SCALE_CFG)[:20];SCALE_RUN.mkdir(parents=True,exist_ok=True)
atomic_json(SCALE_RUN/'config.json',SCALE_CFG)
for k in SCALES:
 q=pd.concat([a[scale_cols] for a in parts_by_scale.pop(k)],ignore_index=True)
 q=q.rename(columns={c:f'S2_{i:02d}' for i,c in enumerate(ANNUAL[:14],1)})
 for forest,part in q.groupby('site'):
  part=part.sort_values(['row','col']).reset_index(drop=True)
  atomic_parquet(part,SCALE_RUN/f'{forest}_{k}m.parquet')
 del q
del parts_by_scale,J50
assert all((SCALE_RUN/f'{s}_{k}m.parquet').is_file() for s in SITES for k in SCALES)
display(D.groupby('site').target_fraction.agg(n='size',mean='mean',std='std',max='max').round(3))
print('Complete native V5 reference:',REF)


## 04 — Within-landscape spatial-block modelling, all eligible training data

In [ ]:
# All eligible cells, fixed spatial holdout, no LOFO.
from scipy.spatial import cKDTree
from sklearn.ensemble import RandomForestRegressor,ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import joblib
B21=500.;BUFFER21=100.;FOLDS21=5;SEED21=20261123
F21={'annual':ANNUAL[:14],'M1_S2':S2,'M2_EMB':EMB,'M3_S2_EMB':S2+EMB}
M21=D.copy()
M21['block21']=M21.site+'_'+M21.x.floordiv(B21).astype(int).astype(str)+'_'+M21.y.floordiv(B21).astype(int).astype(str)
mapping={};rng=np.random.default_rng(SEED21)
for forest,g in M21.groupby('site',sort=False):
 counts=g.groupby('block21').size();order=rng.permutation(len(counts));order=order[np.argsort(-counts.to_numpy()[order],kind='stable')]
 sizes=np.zeros(5,int)
 for i in order:
  k=int(sizes.argmin());mapping[counts.index[i]]=k;sizes[k]+=counts.iloc[i]
M21['fold21']=M21.block21.map(mapping).astype(int)
assert M21.groupby('block21').fold21.nunique().eq(1).all()
assert M21.groupby('site').fold21.nunique().eq(5).all()
print('Spatial folds:');display(M21.groupby(['site','fold21']).size().unstack())
def model_factory(name):
 if name=='HGB':return HistGradientBoostingRegressor(random_state=SEED21,**MODEL)
 if name=='RF':return RandomForestRegressor(n_estimators=120,min_samples_leaf=5,max_features=.7,n_jobs=2,random_state=SEED21)
 if name=='ExtraTrees':return ExtraTreesRegressor(n_estimators=160,min_samples_leaf=5,max_features=.8,n_jobs=2,random_state=SEED21)
 if name=='Ridge':return make_pipeline(StandardScaler(),Ridge(alpha=10.))
 raise ValueError(name)
def weights(g,strategy):
 if strategy in ['natural','local']:return np.ones(len(g))
 w=1/g.site.map(g.site.value_counts()).to_numpy(float)
 if strategy=='forest_abundance':
  w*=np.where(g.target_fraction>=.4,1.5,1.)
  for s in g.site.unique():w[g.site.eq(s)]/=w[g.site.eq(s)].sum()
 return w/w.mean()
def buffered_splits(d):
 coords=d[['x','y']].to_numpy(float)
 for k in range(5):
  te=np.flatnonzero(d.fold21.to_numpy()==k);tr=np.flatnonzero(d.fold21.to_numpy()!=k)
  keep=np.ones(len(tr),bool)
  for s in d.site.unique():
   a=np.flatnonzero(d.site.iloc[tr].to_numpy()==s);b=te[d.site.iloc[te].to_numpy()==s]
   if len(a) and len(b):keep[a]=cKDTree(coords[b]).query(coords[tr[a]])[0]>=BUFFER21
  tr=tr[keep];assert len(tr) and len(te)
  yield k,tr,te
def fit_est(name,features,d,tr,te,strategy):
 g=d.iloc[tr];m=model_factory(name);X=g[features].to_numpy(np.float32);y=g.target_fraction.to_numpy(float);w=weights(g,strategy)
 if name=='Ridge':m.fit(X,y,ridge__sample_weight=w)
 else:m.fit(X,y,sample_weight=w)
 return np.clip(m.predict(d.iloc[te][features].to_numpy(np.float32)),0,1)
def evaluate_spatial(d,sets,experiments,label):
 d=d.reset_index(drop=True).copy()
 assert d.fold21.notna().all() and d.groupby('site').fold21.nunique().eq(5).all()
 cols=sorted(set(sum(sets.values(),[])))
 assert np.isfinite(d[cols].to_numpy(float)).all()
 cfg=dict(version='within_landscape_v2',source=dataframe_sha(d[['site','x','y','target_fraction','fold21']+cols]),features=sets,experiments=experiments,seed=SEED21,buffer=BUFFER21,block=B21,hgb=MODEL,rf=dict(n=120,leaf=5,features=.7),extra=dict(n=160,leaf=5,features=.8),ridge=10)
 dest=OUT/label/digest(cfg)[:20];dest.mkdir(parents=True,exist_ok=True);atomic_json(dest/'config.json',cfg)
 pred=d[['site','x','y','row','col','target_fraction','block21','fold21']].copy()
 audits=[]
 for k,tr,te in buffered_splits(d):
  audits.append(dict(fold=k,train=len(tr),test=len(te),**{s+'_train':int(d.site.iloc[tr].eq(s).sum()) for s in SITES}))
  for family,alg,strategy in experiments:
   name=f'{family}__{alg}__{strategy}';p=dest/f'{name}_fold{k}.npz'
   if p.exists():
    with np.load(p,allow_pickle=False) as z:
     assert np.array_equal(z['test'],te);v=z['pred']
   elif strategy=='local':
    v=np.full(len(te),np.nan)
    for s in d.site.unique():
     a=tr[d.site.iloc[tr].to_numpy()==s];b=np.flatnonzero(d.site.iloc[te].to_numpy()==s)
     if len(b):assert len(a)>0,(s,k);v[b]=fit_est(alg,sets[family],d,a,te[b],strategy)
   else:v=fit_est(alg,sets[family],d,tr,te,strategy)
   assert np.isfinite(v).all() and len(v)==len(te)
   if not p.exists():
    tmp=p.with_suffix('.partial')
    with open(tmp,'wb') as f:np.savez_compressed(f,test=te,pred=v)
    os.replace(tmp,p)
   pred.loc[te,name]=v
   print(label,k,name,flush=True)
 pd.DataFrame(audits).to_csv(dest/'fold_training_counts.csv',index=False)
 names=[f'{a}__{b}__{c}' for a,b,c in experiments]
 rows=[];summary=[]
 mu=pred.groupby('site').target_fraction.mean().mean()
 variance=np.mean([np.mean((g.target_fraction-mu)**2) for _,g in pred.groupby('site')])
 for name in names:
  for s,g in pred.groupby('site'):
   y=g.target_fraction.to_numpy();p=g[name].to_numpy();e=p-y;hi=y>=.4
   rows.append(dict(model=name,site=s,n=len(g),R2=1-np.mean(e**2)/np.var(y) if np.var(y)>0 else np.nan,MAE_pp=100*np.mean(abs(e)),RMSE_pp=100*np.sqrt(np.mean(e**2)),bias_pp=100*np.mean(e),high_n=int(hi.sum()),high_bias_pp=100*np.mean(e[hi]) if hi.any() else np.nan,high_recall=np.mean(p[hi]>=.4) if hi.any() else np.nan))
  g=pd.DataFrame(rows);g=g[g.model.eq(name)]
  mse=np.mean((g.RMSE_pp/100)**2)
  summary.append(dict(model=name,n=len(pred),equal_forest_R2=1-mse/variance if variance>0 else np.nan,MAE_pp=g.MAE_pp.mean(),RMSE_pp=100*np.sqrt(mse),bias_pp=g.bias_pp.mean()))
 per=pd.DataFrame(rows);summ=pd.DataFrame(summary).sort_values('MAE_pp')
 atomic_parquet(pred,dest/'heldout_predictions.parquet');per.to_csv(dest/'by_forest.csv',index=False);summ.to_csv(dest/'summary.csv',index=False)
 display(summ.round(3));return pred,per,summ,dest
EXPERIMENTS=[(family,alg,'forest_equal') for family in F21 for alg in ['HGB','RF','ExtraTrees','Ridge']]
EXPERIMENTS += [(family,'HGB',strategy) for family in ['M1_S2','M2_EMB','M3_S2_EMB'] for strategy in ['natural','forest_abundance','local']]
PNEW,PERNEW,SUMNEW,RUNNEW=evaluate_spatial(M21,F21,EXPERIMENTS,'within_landscape_50m')
display(PERNEW.round(3))
# Paired block bootstrap for improvement over prespecified all-data equal-forest HGB.
BASELINE='M3_S2_EMB__HGB__forest_equal'
ci=[];rng=np.random.default_rng(SEED21)
for name in SUMNEW.model:
 dist=[]
 for s,g in PNEW.groupby('site'):
  a=g.assign(delta=100*(abs(g[name]-g.target_fraction)-abs(g[BASELINE]-g.target_fraction))).groupby('block21').delta.agg(['sum','count']).to_numpy(float)
  draws=a[rng.integers(0,len(a),(1000,len(a)))].sum(axis=1);dist.append(draws[:,0]/draws[:,1])
 v=np.mean(dist,axis=0);ci.append(dict(model=name,delta_MAE_pp=float(SUMNEW.set_index('model').loc[name,'MAE_pp']-SUMNEW.set_index('model').loc[BASELINE,'MAE_pp']),ci_low=np.quantile(v,.025),ci_high=np.quantile(v,.975)))
pd.DataFrame(ci).to_csv(RUNNEW/'paired_block_bootstrap.csv',index=False);display(pd.DataFrame(ci).round(3))
# Retained spectral figure interfaces: fixed HGB equal-forest predictions, no LOFO.
P=PNEW.rename(columns={f'{f}__HGB__forest_equal':f for f in ['M1_S2','M2_EMB','M3_S2_EMB']}).copy()
P['M0_mean']=np.nan
for k,tr,te in buffered_splits(M21):P.loc[te,'M0_mean']=np.average(M21.target_fraction.iloc[tr],weights=weights(M21.iloc[tr],'forest_equal'))
RUN=RUNNEW


In [ ]:
MODELS=['M0_mean',*FEATURE_SETS]
def metrics(g,col):
 obs=g.target_fraction.to_numpy(float)
 pred=g[col].to_numpy(float)
 err=pred-obs
 var=float(np.mean((obs-np.mean(obs))**2))
 rho=spearmanr(obs,pred).statistic if np.std(pred)>1e-12 and np.std(obs)>1e-12 else np.nan
 return dict(n=len(g),R2=float(1-np.mean(err**2)/var) if var>0 else np.nan,
             MAE_pp=float(100*np.mean(np.abs(err))),
             RMSE_pp=float(100*np.sqrt(np.mean(err**2))),
             bias_pp=float(100*np.mean(err)),
             spearman_rho=float(rho) if np.isfinite(rho) else np.nan)
PER=pd.DataFrame([dict(site=forest,model=model,**metrics(g,model))
 for forest,g in P.groupby('site') for model in MODELS])
PER.to_csv(RUN/'02_continuous_by_forest.csv',index=False)
SUMMARY=[]
for model in MODELS:
 sub=PER[PER.model.eq(model)]
 # Global equal-forest R² with pooled equal-forest target variance.
 means=P.groupby('site').target_fraction.mean()
 v=np.mean([np.mean((g.target_fraction-means[s])**2)+(means[s]-means.mean())**2
      for s,g in P.groupby('site')])
 mse=np.mean([np.mean((g[model]-g.target_fraction)**2) for s,g in P.groupby('site')])
 SUMMARY.append(dict(model=model,n_cells=len(P),n_forests=len(SITES),
  equal_forest_R2=float(1-mse/v) if v>0 else np.nan,
  equal_forest_MAE_pp=float(sub.MAE_pp.mean()),
  equal_forest_RMSE_pp=float(np.sqrt(np.mean(sub.RMSE_pp**2))),
  equal_forest_bias_pp=float(sub.bias_pp.mean()),
  mean_forest_rho=float(sub.spearman_rho.mean())))
SUMMARY=pd.DataFrame(SUMMARY)
SUMMARY.to_csv(RUN/'02_equal_forest_summary.csv',index=False)
DISC=[]
for (forest,model),g in (( (f,m),q) for f,q in P.groupby('site') for m in MODELS):
 obs=g.target_fraction.to_numpy(float)
 pred=g[model].to_numpy(float)
 for t in THRESHOLDS:
  true=(obs>=t).astype(int)
  npos=int(true.sum());nneg=len(true)-npos
  auc=roc_auc_score(true,pred) if npos and nneg and np.unique(pred).size>1 else np.nan
  ap=average_precision_score(true,pred) if npos and nneg else np.nan
  DISC.append(dict(site=forest,model=model,threshold=t,
       n=len(g),positive_n=npos,negative_n=nneg,
       prevalence=npos/len(g),ROC_AUC=auc,average_precision=ap))
DISC=pd.DataFrame(DISC)
DISC.to_csv(RUN/'02_abundance_detection_by_forest.csv',index=False)
DISPLAY_DISC=(DISC.groupby(['model','threshold']).agg(
 n_forests_evaluable=('ROC_AUC','count'),
 mean_AUC=('ROC_AUC','mean'),
 mean_AP=('average_precision','mean'),
 mean_prevalence=('prevalence','mean')).reset_index())
DISPLAY_DISC.to_csv(RUN/'02_equal_forest_detection_summary.csv',index=False)
print('Equal-forest continuous-cover scores:')
display(SUMMARY.round(3))
print('Abundance discrimination (undefined sites omitted, counts explicit):')
display(DISPLAY_DISC.round(3))
print('Important: high-cover detection is undefined for forests without high-cover examples.')


## 05 — Resolution sensitivity, 30 and 50 m annual S2

In [ ]:
# 30/50 m: same spatial block assignment, annual predictors on common family support within each scale.
# 30m blocks not represented at 50m receive deterministic fold IDs independent of response.
for scale in [30,50]:
 parts=[]
 for forest in SITES:
  g=pd.read_parquet(SCALE_RUN/f'{forest}_{scale}m.parquet').rename(columns={f'S2_{i:02d}':c for i,c in enumerate(ANNUAL[:14],1)})
  parts.append(g)
 z=pd.concat(parts,ignore_index=True)
 z['block21']=z.site+'_'+z.x.floordiv(B21).astype(int).astype(str)+'_'+z.y.floordiv(B21).astype(int).astype(str)
 z['fold21']=z.block21.map(lambda b:mapping.get(b,int(hashlib.sha256(b.encode()).hexdigest()[:8],16)%5))
 # Annual-only matched predictor recipe; each resolution has its own valid UAV support.
 a,b,c,dest=evaluate_spatial(z,{'annual':ANNUAL[:14]},[('annual','HGB','forest_equal'),('annual','HGB','local')],f'within_landscape_scale_{scale}m')
 print('Resolution',scale,'eligible cells',len(z),'results',dest)


## 06 — Spectral interpretation and held-out abundance discrimination

In [ ]:
import matplotlib
from matplotlib.lines import Line2D
# A: Within-forest annual S2 contrast. Use percentiles for robustness.
low=.05;high=.40
rows=[]
for forest,g in D.groupby('site'):
 lo=g[g.target_fraction<=low]
 hi=g[g.target_fraction>=high]
 if len(lo)<10 or len(hi)<10:
  print('Spectral contrast not estimable',forest,'low',len(lo),'high',len(hi));continue
 for i,c in enumerate(ANNUAL,1):
  lv=float(lo[c].median());hv=float(hi[c].median())
  rows.append(dict(site=forest,band=c,band_order=i,n_low=len(lo),
                   n_high=len(hi),median_low=lv,median_high=hv,
                   difference=hv-lv))
SPECTRA=pd.DataFrame(rows,columns=['site','band','band_order','n_low','n_high','median_low','median_high','difference'])
SPECTRA.to_csv(RUN/'03_within_forest_annual_S2_spectra.csv',index=False)
fig,axs=plt.subplots(2,2,figsize=(12,8),layout='constrained')
for ax,forest in zip(axs.flat,SITES):
 g=SPECTRA[SPECTRA.site.eq(forest)]
 if len(g):
  ax.plot(g.band_order,g.median_low,'o-',label='≤5% UAV Musanga')
  ax.plot(g.band_order,g.median_high,'o-',label='≥40% UAV Musanga')
  ax.set_title(f'{forest} | low n={g.n_low.iloc[0]}, high n={g.n_high.iloc[0]}')
 else:
  ax.set_title(f'{forest} | too few high/low cells')
 ax.set(xlabel='Annual S2 source band number',ylabel='Median annual band value')
 ax.grid(alpha=.15)
axs.flat[0].legend(fontsize=8)
fig.suptitle('A · Within-forest spectral associations (not species-specific identification)')
fig.savefig(RUN/'03A_S2_within_forest_low_high_spectra.png',dpi=170,bbox_inches='tight')
plt.show();plt.close(fig)
# B: For each forest show S2 vs embeddings+S2; cap plots only, not metrics.
fig,axs=plt.subplots(2,2,figsize=(11,10),layout='constrained')
for ax,forest in zip(axs.flat,SITES):
 g=P[P.site.eq(forest)].copy()
 if len(g)>1300:g=g.sample(n=1300,random_state=SEED)
 ax.scatter(g.target_fraction*100,g.M1_S2*100,s=9,alpha=.3,label='S2')
 ax.scatter(g.target_fraction*100,g.M3_S2_EMB*100,s=9,alpha=.3,label='S2 + embeddings')
 ax.plot([0,100],[0,100],'k--',lw=1)
 ax.set(title=forest,xlabel='UAV Musanga cover (%)',
        ylabel='Held-out satellite prediction (%)',
        xlim=(-2,102),ylim=(-2,102))
 ax.grid(alpha=.15)
axs.flat[0].legend(frameon=False)
fig.suptitle('B · Spatial-block held-out prediction (matched reference cells)')
fig.savefig(RUN/'03B_heldout_scatter_by_forest.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)
# C: Mean AUC by cover threshold and number of forests actually evaluable.
fig,ax=plt.subplots(figsize=(8,5))
for model in MODELS[1:]:
 q=DISPLAY_DISC[DISPLAY_DISC.model.eq(model)].sort_values('threshold')
 ax.plot(q.threshold*100,q.mean_AUC,marker='o',label=model)
 for v in q.itertuples():
  if np.isfinite(v.mean_AUC):
   ax.annotate(f'n={v.n_forests_evaluable}',(v.threshold*100,v.mean_AUC),
               xytext=(0,8),textcoords='offset points',ha='center',fontsize=8)
ax.axhline(.5,ls='--',color='gray',lw=1,label='Chance AUC')
ax.set(xlabel='UAV Musanga cover threshold (%)',ylabel='Forest-mean held-out ROC-AUC',
       ylim=(0,1.05),xticks=[5,20,40],
       title='C · Ability to distinguish Musanga-abundant patches')
ax.legend(fontsize=9);ax.grid(alpha=.15)
fig.savefig(RUN/'03C_abundance_discrimination.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)
print('Three figures saved:',RUN)


In [ ]:
# 10 original annual reflectance bands (verified upstream Earth Engine selection order)
S2_BANDS=['B2','B3','B4','B8','B5','B6','B7','B8A','B11','B12']
S2_WAVELENGTH_NM=np.array([492,560,665,833,704,740,783,865,1610,2190])
S2_ORDER=np.argsort(S2_WAVELENGTH_NM)
assert len(ANNUAL)==15 and len(QUARTER)==44
REFLECTANCE=[f'RAW_ANNUAL__{i:02d}' for i in range(1,11)]
INDICES=[f'RAW_ANNUAL__{i:02d}' for i in range(11,15)]
CLEAR_ANNUAL='RAW_ANNUAL__15'
Q_REFLECTANCE={q:[f'S2_QUARTERLY_3YR__{(q-1)*11+i:02d}' for i in range(1,11)] for q in range(1,5)}
assert set(REFLECTANCE+INDICES+[CLEAR_ANNUAL])==set(ANNUAL)
assert set(sum(Q_REFLECTANCE.values(),[])+Q_COUNTS)==set(QUARTER)
# Plot median and interquartile envelope of raw reflectance; no modelling in this block.
spectral_rows=[]
fig,axarr=plt.subplots(2,2,figsize=(15,10),layout='constrained')
for ax,forest in zip(axarr.flat,SITES):
 g=D[D.site.eq(forest)]
 low=g[g.target_fraction<=.05];high=g[g.target_fraction>=.40]
 if len(high)<10 or len(low)<10:
  ax.text(.5,.5,f'Insufficient contrast\\nlow={len(low)}, high={len(high)}',
          ha='center',va='center',transform=ax.transAxes)
  ax.set_title(f'{forest} — not estimable')
  continue
 for subset,label,color in [(low,'≤5% Musanga','steelblue'),(high,'≥40% Musanga','darkorange')]:
  arr=subset[REFLECTANCE].to_numpy(float)[:,S2_ORDER]
  wl=S2_WAVELENGTH_NM[S2_ORDER]
  med=np.median(arr,axis=0);q25=np.percentile(arr,25,axis=0);q75=np.percentile(arr,75,axis=0)
  ax.plot(wl,med,'o-',label=f'{label}, n={len(subset)}',color=color)
  ax.fill_between(wl,q25,q75,alpha=.16,color=color)
  for j,k in enumerate(S2_ORDER):
   spectral_rows.append(dict(site=forest,group=label,band=S2_BANDS[k],
     wavelength_nm=int(S2_WAVELENGTH_NM[k]),n=len(subset),
     median=float(med[j]),q25=float(q25[j]),q75=float(q75[j])))
 ax.set(xlabel='Approximate central wavelength (nm)',
        ylabel='S2 annual median surface reflectance',title=forest)
 ax.legend(fontsize=8);ax.grid(alpha=.12)
fig.suptitle('True annual S2 reflectance: low vs high 50 m Musanga cover (median ± IQR)')
fig.savefig(RUN/'05A_annual_S2_true_reflectance_IQR.png',dpi=180)
plt.show();plt.close(fig)
pd.DataFrame(spectral_rows).to_csv(RUN/'05A_annual_S2_true_reflectance_IQR.csv',index=False)
# Seasonal reflectance: 3-year pooled quarter medians, NOT single-year snapshots.
seasonal_rows=[]
fig,axarr=plt.subplots(2,2,figsize=(14,9),layout='constrained')
for ax,forest in zip(axarr.flat,SITES):
 g=D[D.site.eq(forest)]
 low=g[g.target_fraction<=.05];high=g[g.target_fraction>=.40]
 if len(high)<10 or len(low)<10:
  ax.text(.5,.5,f'Insufficient contrast: high={len(high)}',ha='center',va='center',transform=ax.transAxes)
  ax.set_title(forest);continue
 for q in range(1,5):
  for subset,name in [(low,'low'),(high,'high')]:
   # NDVI is computed from quarter median reflectances (B8 = index 4; B4 = index 3).
   nir=subset[Q_REFLECTANCE[q][3]].to_numpy(float)
   red=subset[Q_REFLECTANCE[q][2]].to_numpy(float)
   ndvi=(nir-red)/np.maximum(nir+red,1e-8)
   med=float(np.median(ndvi))
   seasonal_rows.append(dict(site=forest,quarter=q,class_name=name,
                              n=len(subset),median_NDVI=med,
                              q25_NDVI=float(np.percentile(ndvi,25)),
                              q75_NDVI=float(np.percentile(ndvi,75))))
 for name,color in [('low','steelblue'),('high','darkorange')]:
  a=pd.DataFrame(seasonal_rows)
  a=a[(a.site==forest)&(a.class_name==name)].sort_values('quarter')
  ax.plot(a.quarter,a.median_NDVI,'o-',color=color,label=name)
  ax.fill_between(a.quarter.to_numpy(float),a.q25_NDVI.to_numpy(float),
                  a.q75_NDVI.to_numpy(float),alpha=.15,color=color)
 ax.set(title=forest,xlabel='Pooled 3-year calendar quarter',ylabel='Quarterly median-reflectance NDVI',
        xticks=[1,2,3,4])
 ax.grid(alpha=.15);ax.legend()
fig.suptitle('Seasonal spectral association: quarter medians pooled across three years')
fig.savefig(RUN/'05B_pooled_quarterly_NDVI_IQR.png',dpi=180)
plt.show();plt.close(fig)
pd.DataFrame(seasonal_rows).to_csv(RUN/'05B_pooled_quarterly_NDVI_IQR.csv',index=False)
print('Spectral plots complete. Overlap of distributions is expected in mixed 50 m canopies.')


## 07 — Structure and disturbance controls

In [ ]:
# Optional independent controls, freshly downloaded; no historical table.
# Native 30m TMF and 100m tree cover are sampled on the 10m lattice, not new 10m observations.
STRUCT_CFG=dict(version='v5_structure_2020_v1',height='users/nlang/ETH_GlobalCanopyHeight_2020_10m_v1',height_sd='users/nlang/ETH_GlobalCanopyHeightSD_2020_10m_v1',tmf='projects/JRC/TMF/v1_2025/AnnualChanges',tmf_band='Dec2020',classes=[1,2,4],tree='COPERNICUS/Landcover/100m/Proba-V-C3/Global/2019',tree_band='tree-coverfraction',aggregation='50m cell means and height SD; coarse source nearest sampling at 10m',grid=SAT_ID)
STRUCT_ID=digest(STRUCT_CFG)[:20];STRUCT_DIR=ROOT/'structure_10m'/STRUCT_ID;STRUCT_DIR.mkdir(parents=True,exist_ok=True)
atomic_json(STRUCT_DIR/'config.json',STRUCT_CFG)
CONTROLS14=['height_local_mean_m','height_local_sd_m','tmf_undisturbed_local_frac','tmf_degraded_local_frac','tmf_regrowth_local_frac','cop_2019_tree_fraction_pct']
STRUCT_COLS12=CONTROLS14+['height_model_sd_local_m']
BANDS14=S2_BANDS;spec14=dict(zip(BANDS14,ANNUAL[:10]))
STRUCT_STATUS='disabled';D14=pd.DataFrame()
if RUN_STRUCTURE:
 try:
  st_parts=[]
  for (forest,tx,ty),d in D.groupby(['site','tile_x','tile_y'],sort=True):
   tx=int(tx);ty=int(ty)
   h=ee.Image(STRUCT_CFG['height']).select([0]).rename('height')
   sd=ee.Image(STRUCT_CFG['height_sd']).select([0]).rename('height_model_sd')
   tmf=ee.ImageCollection(STRUCT_CFG['tmf']).mosaic().select(STRUCT_CFG['tmf_band']).rename('tmf')
   tree=ee.Image(STRUCT_CFG['tree']).select(STRUCT_CFG['tree_band']).rename('tree')
   bands=['height','height_model_sd','tmf','tree']
   a=fetch_image(ee.Image.cat([h,sd,tmf,tree]).unmask(-9999,sameFootprint=False).toFloat(),bands,STRUCT_DIR,forest,tx,ty,'STRUCT',STRUCT_ID)
   b=tile_bounds(tx,ty);rr=np.floor((b[3]-d.y.to_numpy())/50).astype(int);cc=np.floor((d.x.to_numpy()-b[0])/50).astype(int)
   good=(a!=-9999).all(axis=0)&np.isfinite(a).all(axis=0)&(a[0]>=0)&(a[1]>=0)&np.isin(a[2],[1,2,3,4,5,6])&(a[3]>=0)&(a[3]<=100)
   count=reduce_grid(good.astype(np.int32),50,10)
   def mean(z):
    total=reduce_grid(np.where(good,z,0.),50,10)
    return np.divide(total,count,out=np.full(total.shape,np.nan,dtype=np.float32),where=count>0)
   mh=mean(a[0]);height_sd=np.sqrt(np.maximum(mean(a[0]**2)-mh**2,0))
   fields=[mh,height_sd,mean(a[2]==1),mean(a[2]==2),mean(a[2]==4),mean(a[3]),mean(a[1])]
   z=d.copy().reset_index(drop=True)
   for name,v in zip(STRUCT_COLS12,fields):z[name]=v[rr,cc]
   z['structure_valid_fraction']=count[rr,cc]/25.
   st_parts.append(z[z.structure_valid_fraction.ge(.90)])
  D14=pd.concat(st_parts,ignore_index=True).dropna(subset=STRUCT_COLS12)
  atomic_parquet(D14,ROOT/f'v5_structure_50m_{SIGNATURE}_{STRUCT_ID}.parquet')
  STRUCT_STATUS='complete'
  display(D14.groupby('site').size().rename('valid_structure_cells'))
 except Exception as exc:
  STRUCT_STATUS='failed_optional_access_or_extraction'
  print('OPTIONAL STRUCTURE ANALYSIS UNAVAILABLE:',type(exc).__name__,str(exc)[:500],flush=True)
  atomic_json(RUN/'structure_error.json',dict(status=STRUCT_STATUS,error=str(exc)))
  D14=pd.DataFrame()
atomic_json(RUN/'structure_status.json',dict(status=STRUCT_STATUS,recipe=STRUCT_CFG))


In [ ]:
if STRUCT_STATUS=='complete':
 z=D14.merge(M21[['site','row','col','block21','fold21']],on=['site','row','col'],validate='one_to_one')
 if z.groupby('site').fold21.nunique().eq(5).all():
  sets={'structure':STRUCT_COLS12,'annual':ANNUAL[:14],'structure_annual':STRUCT_COLS12+ANNUAL[:14]}
  evaluate_spatial(z,sets,[(s,'HGB','forest_equal') for s in sets],'within_landscape_structure')


## 08 — Balance-qualified high/low spectral matching

In [ ]:
if STRUCT_STATUS=='complete' and len(D14)>0:
    from scipy.optimize import linear_sum_assignment
    from scipy.spatial.distance import cdist
    SEED17=20261120
    SPECS17={'strict':dict(height=5.,tree=10.,regrowth=.20,dist=2.0),
             'sensitivity':dict(height=8.,tree=15.,regrowth=.30,dist=3.0)}
    CONT17=CONTROLS14
    OUT17=RUN/'balance_qualified_matching'
    OUT17.mkdir(parents=True,exist_ok=True)
    MIN_PAIRS17=15;MIN_BLOCKS17=5;SMD_LIMIT17=.10
    def match17(forest,spec):
     d=D14[D14.site.eq(forest)].reset_index(drop=True)
     if len(d)>2500:d=d.sample(n=2500,random_state=SEED17).reset_index(drop=True)
     h=d[d.target_fraction>=.40].reset_index(drop=True)
     l=d[d.target_fraction<=.05].reset_index(drop=True)
     info=dict(site=forest,specification=spec,n_high=len(h),n_low=len(l),n_initial=0,n_retained=0,
               n_blocks=0,max_abs_smd=np.nan,accepted=False,reason='')
     if len(h)==0 or len(l)==0:
      info['reason']='No high/low common abundance classes';return [],info,[]
     source=d[CONT17].to_numpy(float);sd=source.std(axis=0,ddof=1)
     active=sd>1e-10;sd=np.where(active,sd,1.)
     if not active.all():
      info['reason']='One or more constant control covariates; overlap not established';return [],info,[]
     zh=h[CONT17].to_numpy(float)/sd;zl=l[CONT17].to_numpy(float)/sd
     dist=cdist(zh,zl)
     rule=SPECS17[spec]
     spatial=cdist(h[['x','y']],l[['x','y']])
     eligible=(np.abs(h.height_local_mean_m.to_numpy()[:,None]-l.height_local_mean_m.to_numpy()[None,:])<=rule['height'])
     eligible&=(np.abs(h.cop_2019_tree_fraction_pct.to_numpy()[:,None]-l.cop_2019_tree_fraction_pct.to_numpy()[None,:])<=rule['tree'])
     eligible&=(np.abs(h.tmf_regrowth_local_frac.to_numpy()[:,None]-l.tmf_regrowth_local_frac.to_numpy()[None,:])<=rule['regrowth'])
     eligible&=(dist<=rule['dist'])&(spatial>=150.)
     # Dummy columns allow high cells to remain unmatched.
     penalty=10000.
     cost=np.concatenate((np.where(eligible,dist,penalty*2),np.full((len(h),len(h)),penalty)),axis=1)
     ii,jj=linear_sum_assignment(cost)
     pairs=[(int(i),int(j)) for i,j in zip(ii,jj) if j<len(l) and eligible[i,j]]
     info['n_initial']=len(pairs)
     # Outcome-blind balance trimming: remove the pair most contributing to the most imbalanced mean.
     # Never use spectral measurements, and stop before the minimum required sample size.
     while len(pairs)>MIN_PAIRS17:
      differences=np.array([zh[i]-zl[j] for i,j in pairs])
      smd=differences.mean(axis=0)
      if np.max(np.abs(smd))<=SMD_LIMIT17:break
      worst=int(np.argmax(np.abs(smd)))
      contributions=np.sign(smd[worst])*differences[:,worst]
      pairs.pop(int(np.argmax(contributions)))
     if pairs:
      diffs=np.array([zh[i]-zl[j] for i,j in pairs])
      smd=diffs.mean(axis=0)
      info['max_abs_smd']=float(np.max(np.abs(smd)))
     else:smd=np.full(len(CONT17),np.nan)
     rows=[];blocks=set()
     for ix,(i,j) in enumerate(pairs):
      a=h.iloc[i];b=l.iloc[j]
      block=f"{forest}_{int(a.x//500)}_{int(a.y//500)}";blocks.add(block)
      record=dict(site=forest,specification=spec,pair=ix,block500=block,
                  high_cover=float(a.target_fraction),low_cover=float(b.target_fraction),
                  high_x=float(a.x),high_y=float(a.y),low_x=float(b.x),low_y=float(b.y))
      for feature in CONT17:
       record['high_'+feature]=float(a[feature]);record['low_'+feature]=float(b[feature])
      for band,col in spec14.items():record['delta_'+band]=float(a[col]-b[col])
      rows.append(record)
     info['n_retained']=len(rows);info['n_blocks']=len(blocks)
     info['accepted']=bool(len(rows)>=MIN_PAIRS17 and len(blocks)>=MIN_BLOCKS17 and np.all(np.abs(smd)<=SMD_LIMIT17))
     info['reason']='Balance and support pass' if info['accepted'] else 'Failed balance and/or support; spectral attribution inconclusive'
     balance=[dict(site=forest,specification=spec,control=feature,smd=float(v),passes=bool(abs(v)<=SMD_LIMIT17)) for feature,v in zip(CONT17,smd)] if rows else []
     return rows,info,balance
    rows17=[];audit17=[];bal17=[]
    for setting in SPECS17:
     for forest in SITES:
      r,a,b=match17(forest,setting);rows17.extend(r);audit17.append(a);bal17.extend(b)
    PAIRS17=pd.DataFrame(rows17);AUDIT17=pd.DataFrame(audit17);BAL17=pd.DataFrame(bal17)
    AUDIT17.to_csv(OUT17/'17_matching_attrition.csv',index=False)
    BAL17.to_csv(OUT17/'17_covariate_balance.csv',index=False)
    if len(PAIRS17):PAIRS17.to_parquet(OUT17/'17_matched_pairs.parquet',index=False)
    display(AUDIT17.round(3))
    if len(BAL17):display(BAL17.pivot_table(index=['site','specification'],columns='control',values='smd').round(3))
    
    def ci18(df,col,reps=2000,seed=20261120):
     groups=[g[col].to_numpy(float) for _,g in df.groupby('block500')]
     if len(groups)<MIN_BLOCKS17:return np.nan,np.nan
     sums=np.array([g.sum() for g in groups]);counts=np.array([len(g) for g in groups])
     rng=np.random.default_rng(seed);take=rng.integers(0,len(groups),(reps,len(groups)))
     dist=sums[take].sum(axis=1)/counts[take].sum(axis=1)
     return np.quantile(dist,[.025,.975])
    spectral18=[]
    for item in AUDIT17.itertuples():
     if not item.accepted:continue
     f=PAIRS17[(PAIRS17.site==item.site)&(PAIRS17.specification==item.specification)]
     for band in BANDS14:
      col='delta_'+band;low,high=ci18(f,col,seed=SEED17+sum(map(ord,item.site))+BANDS14.index(band))
      spectral18.append(dict(site=item.site,specification=item.specification,band=band,n_pairs=len(f),
          n_blocks=f.block500.nunique(),difference=float(f[col].mean()),ci_low=low,ci_high=high))
    SPECTRAL18=pd.DataFrame(spectral18,columns=['site','specification','band','n_pairs','n_blocks','difference','ci_low','ci_high'])
    SPECTRAL18.to_csv(OUT17/'18_balance_qualified_spectral.csv',index=False)
    if SPECTRAL18.empty:print('NO BALANCE-QUALIFIED SPECTRAL CONTRAST. NB08 previous B8 association is not confirmed after adequate balancing.')
    else:display(SPECTRAL18[SPECTRAL18.band.eq('B8')].round(5))
else:print("Independent controls missing; matched spectra not estimated.",flush=True)


## 09 — MOS continuous conditional spectral response and slope test

In [ ]:
mos=D14[D14.site.eq('MOS')] if len(D14) else pd.DataFrame()
ready22=(len(mos)>=100 and (mos.target_fraction>=.4).sum()>=30 and mos.target_fraction.quantile(.01)<=.05 and mos.target_fraction.quantile(.99)>.65 and len(mos.groupby([mos.x.floordiv(500),mos.y.floordiv(500)]))>=15) if len(mos) else False
if ready22:
    # SECTION 22 — Continuous structure-controlled spectral response at MOS
    # Adjusted NIR/red-edge/SWIR curves; spatial-block bootstrap for conditional association.
    from sklearn.preprocessing import SplineTransformer,StandardScaler
    from sklearn.linear_model import Ridge
    from scipy.stats import spearmanr
    assert 'D14' in globals() and 'CONTROLS14' in globals() and 'spec14' in globals(), 'Run native structure extraction first'
    assert 'D4' in globals() and D.equals(D4), 'This must use the V5 NB07 binary-cover reference'
    SITE22='MOS'
    SEED22=20261123
    BOOT22=300
    BLOCK22=500.
    ALPHA22=10.
    KNOTS22=np.array([0.,.10,.20,.40,.60,.80,1.],dtype=float).reshape(-1,1)
    COL22=['B8','B5','B6','B7','B8A','B11','B12']
    COV22=list(CONTROLS14)
    M22=D14[D14.site.eq(SITE22)].copy().replace([np.inf,-np.inf],np.nan)
    M22=M22.dropna(subset=['target_fraction','x','y']+COV22+[spec14[b] for b in COL22]).reset_index(drop=True)
    assert len(M22)>=100 and M22.target_fraction.between(0,1).all()
    assert (M22.target_fraction>=.4).sum()>=30, 'Not enough high-cover MOS cells for continuous response'
    M22['block22']=np.floor(M22.x/BLOCK22).astype('int64').astype(str)+'_'+np.floor(M22.y/BLOCK22).astype('int64').astype(str)
    BLOCKS22=list(M22.groupby('block22').indices.values())
    assert len(BLOCKS22)>=15,('Too few independent spatial blocks',len(BLOCKS22))
    sha22=hashlib.sha256(pd.util.hash_pandas_object(
        M22[['site','mosaic','row','col','target_fraction']+COV22+[spec14[b] for b in COL22]],
        index=False).values.tobytes()).hexdigest()
    CFG22=dict(version='V5_MOS_continuous_conditional_spectral_v1',
        source_sha=SOURCE_SHA,matched_sample_sha=sha22,
        block_m=BLOCK22,bootstraps=BOOT22,ridge_alpha=ALPHA22,
        spline_knots=KNOTS22.ravel().tolist(),covariates=COV22,bands=COL22,
        inferential_scope='conditional association; not species-specific causal attribution')
    RUN22=OUT/'26_NB08_MOS_continuous_spectral_response'/hashlib.sha256(
        json.dumps(CFG22,sort_keys=True).encode()).hexdigest()[:14]
    RUN22.mkdir(parents=True,exist_ok=True)
    config22=RUN22/'22_config.json'
    if config22.exists():assert json.loads(config22.read_text())==CFG22
    else:config22.write_text(json.dumps(CFG22,indent=2))
    # Fixed spline basis and standardization across block-bootstrap replicates.
    # Predict at the same measured covariate mean. This additive model does not
    # impose a monotonic response; flattening must be supported by intervals.
    spline22=SplineTransformer(knots=KNOTS22,degree=2,include_bias=False,
                               extrapolation='constant')
    ycover22=M22.target_fraction.to_numpy(float)
    sp22=spline22.fit_transform(ycover22.reshape(-1,1))
    standard22=StandardScaler()
    xcov22=standard22.fit_transform(M22[COV22].to_numpy(float))
    X22=np.column_stack([sp22,xcov22])
    Y22=M22[[spec14[b] for b in COL22]].to_numpy(float)
    assert np.isfinite(X22).all() and np.isfinite(Y22).all()
    low22=max(0.,float(np.quantile(ycover22,.01)))
    high22=min(1.,float(np.quantile(ycover22,.99)))
    assert high22>.65,(low22,high22,'Insufficient high-cover support')
    grid22=np.linspace(low22,high22,81)
    Xgrid22=np.column_stack([
        spline22.transform(grid22.reshape(-1,1)),
        np.zeros((len(grid22),len(COV22)),dtype=float)])
    def fit22(indices):
        reg=Ridge(alpha=ALPHA22,fit_intercept=True)
        reg.fit(X22[indices],Y22[indices])
        return reg.predict(Xgrid22)
    point22=fit22(np.arange(len(M22)))
    rng22=np.random.default_rng(SEED22)
    boot22=np.empty((BOOT22,len(grid22),len(COL22)),dtype=np.float32)
    for rep in range(BOOT22):
        draw=rng22.integers(0,len(BLOCKS22),size=len(BLOCKS22))
        ids=np.concatenate([BLOCKS22[j] for j in draw])
        boot22[rep]=fit22(ids)
        if (rep+1)%100==0:print('MOS spectral block bootstrap',rep+1,'/',BOOT22,flush=True)
    l22=np.quantile(boot22,.025,axis=0)
    u22=np.quantile(boot22,.975,axis=0)
    rows22=[]
    for k,band in enumerate(COL22):
        for t,value in enumerate(grid22):
            rows22.append(dict(band=band,cover_fraction=value,cover_pct=100*value,
                adjusted_reflectance=point22[t,k],ci_low=l22[t,k],ci_high=u22[t,k]))
    CURVE22=pd.DataFrame(rows22)
    CURVE22.to_csv(RUN22/'22_MOS_adjusted_spectral_response.csv',index=False)
    # Observed-bin diagnostics, not treated as fitted adjusted effects.
    bins22=[-.001,.05,.20,.40,.60,1.001]
    labels22=['0-5','5-20','20-40','40-60','60-100']
    M22['abundance_bin22']=pd.cut(ycover22,bins22,labels=labels22,include_lowest=True)
    BIN22=M22.groupby('abundance_bin22',observed=True).agg(
        n=('target_fraction','size'),mean_cover=('target_fraction','mean'),
        mean_B8=(spec14['B8'],'mean'),sd_B8=(spec14['B8'],'std')).reset_index()
    BIN22.to_csv(RUN22/'22_MOS_observed_abundance_bins.csv',index=False)
    display(BIN22)
    # Contrasts from the same regression curve; uncertainty uses block bootstrap.
    def contrast22(a,b,band='B8'):
        assert low22<=a<=high22 and low22<=b<=high22,(a,b,low22,high22)
        j=COL22.index(band)
        # Interpolate every bootstrap curve on identical cover grid.
        delta=float(np.interp(b,grid22,point22[:,j])-np.interp(a,grid22,point22[:,j]))
        empirical=np.array([np.interp(b,grid22,v[:,j])-np.interp(a,grid22,v[:,j])
                            for v in boot22],dtype=float)
        lo,hi=np.quantile(empirical,[.025,.975])
        return dict(band=band,from_pct=a*100,to_pct=b*100,
                    adjusted_difference=delta,ci_low=lo,ci_high=hi,
                    n=len(M22),n_blocks=len(BLOCKS22))
    CONTRAST22=pd.DataFrame([contrast22(a,b,band)
        for band in COL22 for a,b in [(.05,.40),(.40,.65)]])
    CONTRAST22.to_csv(RUN22/'22_MOS_continuous_response_contrasts.csv',index=False)
    display(CONTRAST22[CONTRAST22.band.eq('B8')].round(5))
    fig,ax=plt.subplots(figsize=(9,6))
    ix=COL22.index('B8')
    ax.plot(grid22*100,point22[:,ix],color='#185e78',label='Structure-adjusted B8')
    ax.fill_between(grid22*100,l22[:,ix],u22[:,ix],alpha=.22,color='#185e78',
                    label='500 m block bootstrap 95% CI')
    ax.scatter(BIN22.mean_cover*100,BIN22.mean_B8,s=BIN22.n.clip(5,100),
               color='#bc762b',edgecolor='black',zorder=4,label='Observed cover-bin means')
    ax.set(xlabel='UAV V5 Musanga cover (%)',ylabel='Sentinel-2 B8 reflectance',
           title='MOS — conditional continuous B8 spectral response')
    ax.grid(alpha=.2);ax.legend()
    fig.savefig(RUN22/'22_MOS_B8_continuous_adjusted_response.png',dpi=240,bbox_inches='tight')
    plt.show();plt.close(fig)
    fig,axs=plt.subplots(2,3,figsize=(14,8),layout='constrained')
    for ax,band in zip(axs.flat,[b for b in COL22 if b!='B8']):
        k=COL22.index(band)
        ax.plot(grid22*100,point22[:,k],color='#185e78')
        ax.fill_between(grid22*100,l22[:,k],u22[:,k],alpha=.2,color='#185e78')
        ax.set(xlabel='Musanga cover (%)',ylabel='Reflectance',title=band)
        ax.grid(alpha=.15)
    fig.savefig(RUN22/'22_MOS_rededge_SWIR_conditional_response.png',
                dpi=220,bbox_inches='tight')
    plt.show();plt.close(fig)
    QA22=dict(n_cells=int(len(M22)),n_blocks=len(BLOCKS22),
              high_cover_cells=int((ycover22>=.4).sum()),
              cover_min=float(ycover22.min()),cover_max=float(ycover22.max()),
              range_plotted=[float(low22),float(high22)],forest=SITE22,
              target='NB07 V5 band 2 fraction',
              caveat='Conditional additive association; controls may be downstream of Musanga; uncertainty does not include UAV segmentation/reference error.')
    (RUN22/'22_interpretation_and_QA.json').write_text(json.dumps(QA22,indent=2))
    print('Section 22 complete:',RUN22)
    print('Interpretation: association conditional on measured structure and disturbance; not causal attribution.')
    
    # SECTION 24 — explicit slope-change test from Section 22 spatial-block replicates
    assert 'boot22' in globals() and 'point22' in globals() and 'grid22' in globals(), 'Run Section 22 first'
    assert 'COL22' in globals() and 'RUN22' in globals() and 'RUN22' in globals()
    SLOPE24=[]
    for band in COL22:
        k=COL22.index(band)
        def slopes(curve):
            a=np.interp(.05,grid22,curve[:,k])
            b=np.interp(.40,grid22,curve[:,k])
            c=np.interp(.65,grid22,curve[:,k])
            lower=(b-a)/.35
            upper=(c-b)/.25
            return lower,upper,upper-lower
        vals=np.array([slopes(curve) for curve in boot22])
        estimate=slopes(point22)
        ci=np.quantile(vals[:,2],[.025,.975])
        SLOPE24.append(dict(band=band,n_blocks=len(BLOCKS22),
            low_slope_per_fraction=estimate[0],
            high_slope_per_fraction=estimate[1],
            high_minus_low_slope=estimate[2],
            ci_low=ci[0],ci_high=ci[1],
            bootstrap_proportion_negative=float(np.mean(vals[:,2]<0))))
    SLOPE24=pd.DataFrame(SLOPE24)
    SLOPE24.to_csv(RUN22/'24_MOS_spatial_bootstrap_slope_change.csv',index=False)
    display(SLOPE24.round(5))
    fig,ax=plt.subplots(figsize=(8,4.7))
    rows=SLOPE24.set_index('band').loc[COL22]
    y=np.arange(len(rows))
    ax.errorbar(rows.high_minus_low_slope,y,
        xerr=np.vstack([rows.high_minus_low_slope-rows.ci_low,
                        rows.ci_high-rows.high_minus_low_slope]),
        fmt='o',capsize=3)
    ax.axvline(0,color='black',linestyle=':')
    ax.set(yticks=y,yticklabels=rows.index,
           xlabel='Adjusted slope difference: high (40–65%) minus low (5–40%)',
           title='MOS — block bootstrap test of spectral response flattening')
    ax.grid(axis='x',alpha=.2)
    fig.savefig(RUN22/'24_MOS_spectral_slope_change.png',dpi=220,bbox_inches='tight')
    plt.show();plt.close(fig)
    print('Section 24 complete: changes relative to Section 22 estimates, not causal effects.')
else:print("Insufficient complete MOS structure/abundance/block support; no conditional slope claim.",flush=True)


## 10 — Planet NICFI: fresh annual 5 m spectral and texture data, matched spatial hold-out

In [ ]:
# NICFI annual spectral + within-cell texture, independent 5m aligned downloads (~4.77m source).
# Yoko 2020 has incomplete monthly coverage: use historical 2021 choice, explicitly flagged.
NICFI_SOURCE='projects/planet-nicfi/assets/basemaps/africa'
NICFI_YEARS={**YEARS,'Yoko':2021}
NICFI_BANDS=['B','G','R','N']
NIC_CFG=dict(version='nicfi_v5_annual_5m_v1',source=NICFI_SOURCE,years=NICFI_YEARS,scale=5,crs=CRS,grid=[0,0],cadence='monthly',min_months=2,texture='within-cell SD and mean absolute neighboring NDVI differences; no cross-cell neighborhood',scale_factor=.0001)
NIC_ID=digest(NIC_CFG)[:20];NIC_DIR=ROOT/'nicfi_5m'/NIC_ID;NIC_DIR.mkdir(parents=True,exist_ok=True);atomic_json(NIC_DIR/'config.json',NIC_CFG)
NIC_STATUS='incomplete';NIC_RUN=None
def nicfi_tile(forest,tx,ty):
 path=NIC_DIR/f'{forest}_{tx}_{ty}.tif';meta=path.with_suffix('.json')
 tf=tile_transform(tx,ty,5);n=TILE_M//5
 def verify():
  with rasterio.open(path) as ds:
   assert ds.shape==(n,n) and ds.count==4 and ds.crs==rasterio.crs.CRS.from_string(CRS) and ds.transform.almost_equals(tf) and ds.nodata==-9999
   assert list(ds.descriptions)==NICFI_BANDS and ds.tags().get('recipe')==NIC_ID
   return ds.read()
 if path.is_file() and meta.is_file():
  assert json.loads(meta.read_text())['recipe']==NIC_ID
  return verify()
 year=NICFI_YEARS[forest]
 coll=ee.ImageCollection(NICFI_SOURCE).filterBounds(ee_region(tx,ty)).filterDate(f'{year}-01-01',f'{year+1}-01-01').filter(ee.Filter.eq('cadence','monthly'))
 ids=coll.aggregate_array('system:index').getInfo();assert len(ids)>=2,(forest,year,ids)
 im=coll.select(NICFI_BANDS).median().multiply(.0001).unmask(-9999,sameFootprint=False).toFloat()
 params=dict(crs=CRS,crs_transform=list(tf)[:6],dimensions=[n,n],format='GEO_TIFF',filePerBand=False)
 for attempt in range(4):
  try:
   resp=requests.get(im.getDownloadURL(params),timeout=(30,300));resp.raise_for_status()
   with MemoryFile(resp.content) as mem,mem.open() as ds:
    assert ds.shape==(n,n) and ds.count==4 and ds.crs==rasterio.crs.CRS.from_string(CRS) and ds.transform.almost_equals(tf)
    a=ds.read().astype(np.float32)
   assert np.isfinite(a).all()
   tmp=path.with_suffix('.partial.tif')
   with rasterio.open(tmp,'w',driver='GTiff',width=n,height=n,count=4,dtype='float32',crs=CRS,transform=tf,nodata=-9999,compress='deflate',tiled=True) as dst:
    dst.write(a);dst.descriptions=tuple(NICFI_BANDS);dst.update_tags(recipe=NIC_ID)
   os.replace(tmp,path);atomic_json(meta,dict(recipe=NIC_ID,source_image_ids=ids,year=year))
   return verify()
  except Exception as exc:
   if attempt==3:raise
   print('NICFI retry',forest,tx,ty,str(exc)[:160]);time.sleep(2**attempt)
NIC_MEAN=['NIC_'+b+'_mean' for b in NICFI_BANDS]+['NIC_NDVI_mean']
NIC_TEXTURE=['NIC_'+b+'_sd' for b in NICFI_BANDS]+['NIC_NDVI_sd','NIC_NDVI_neighbor_absdiff']
try:
 parts=[]
 for (forest,tx,ty),g in M21.groupby(['site','tile_x','tile_y']):
  a=nicfi_tile(forest,int(tx),int(ty));b=tile_bounds(tx,ty)
  rr=np.floor((b[3]-g.y.to_numpy())/50).astype(int);cc=np.floor((g.x.to_numpy()-b[0])/50).astype(int)
  good=np.isfinite(a).all(axis=0)&(a!=-9999).all(axis=0)&(a>=0).all(axis=0)&((a[3]+a[2])>0)
  ndvi=np.divide(a[3]-a[2],a[3]+a[2],out=np.zeros_like(a[3]),where=good)
  stack=np.concatenate([a,ndvi[None]],axis=0)
  rows=[]
  for i,j in zip(rr,cc):
   z=stack[:,i*10:(i+1)*10,j*10:(j+1)*10];v=good[i*10:(i+1)*10,j*10:(j+1)*10]
   if v.mean()<.9:rows.append([np.nan]*11+[v.mean()]);continue
   q=z[:,v];nd=z[-1];dx=abs(np.diff(nd,axis=1))[v[:,1:]&v[:,:-1]];dy=abs(np.diff(nd,axis=0))[v[1:,:]&v[:-1,:]]
   rows.append([*q.mean(axis=1),*q.std(axis=1),np.concatenate([dx,dy]).mean(),v.mean()])
  f=pd.DataFrame(rows,columns=NIC_MEAN+NIC_TEXTURE+['nicfi_support'],index=g.index)
  parts.append(pd.concat([g,f],axis=1));print('NICFI tile',forest,tx,ty,flush=True)
 DNIC=pd.concat(parts).dropna(subset=NIC_MEAN+NIC_TEXTURE).reset_index(drop=True)
 atomic_parquet(DNIC,ROOT/f'nicfi_50m_{SOURCE_SHA[:12]}_{NIC_ID}.parquet')
 print('NICFI common support');display(DNIC.groupby('site').size())
 assert set(DNIC.site)==set(SITES) and DNIC.groupby('site').fold21.nunique().eq(5).all()
 sets={'NICFI_means':NIC_MEAN,'NICFI_texture':NIC_MEAN+NIC_TEXTURE,'S2_EMB':S2+EMB,'S2_EMB_NICFI':S2+EMB+NIC_MEAN+NIC_TEXTURE}
 ex=[(f,alg,'forest_equal') for f in sets for alg in ['HGB','ExtraTrees']]+[(f,'HGB','local') for f in sets]
 PN,PER_N,SUM_N,NIC_RUN=evaluate_spatial(DNIC,sets,ex,'within_landscape_NICFI_50m')
 NIC_STATUS='complete'
 # Spectral means for high/low classes, descriptive; no claim of species specificity.
 contrast=[]
 for s,g in DNIC.groupby('site'):
  lo=g[g.target_fraction<=.05];hi=g[g.target_fraction>=.4]
  if min(len(lo),len(hi))>=10:
   for c in NIC_MEAN+NIC_TEXTURE:contrast.append(dict(site=s,feature=c,n_low=len(lo),n_high=len(hi),median_low=lo[c].median(),median_high=hi[c].median(),difference=hi[c].median()-lo[c].median()))
 pd.DataFrame(contrast).to_csv(NIC_RUN/'NICFI_low_high_contrast.csv',index=False)
except Exception as exc:
 NIC_STATUS='incomplete_access_or_processing'
 print('NICFI not completed:',type(exc).__name__,str(exc)[:500]);atomic_json(NIC_DIR/'error.json',dict(status=NIC_STATUS,error=str(exc)))


## 11 — All-data candidate refit and completion record

In [ ]:
# Candidate all-data refit; CV selection is exploratory, no external transfer claim.
best=SUMNEW[~SUMNEW.model.str.endswith('__local')].iloc[0].model
family,alg,strategy=best.split('__');features=F21[family]
m=model_factory(alg);X=M21[features].to_numpy(np.float32);y=M21.target_fraction.to_numpy(float);w=weights(M21,strategy)
if alg=='Ridge':m.fit(X,y,ridge__sample_weight=w)
else:m.fit(X,y,sample_weight=w)
PROD=RUNNEW/'all_data_candidate';PROD.mkdir(exist_ok=True)
joblib.dump(m,PROD/'model.joblib')
atomic_json(PROD/'model_spec.json',dict(model=best,features=features,n_train=len(M21),source_sha=SOURCE_SHA,scope='within represented landscapes only; exploratory CV selection; predictions clipped 0–1',sklearn_version=__import__('sklearn').__version__))
atomic_json(OUT/'08_within_landscape_completion.json',dict(primary_complete=True,reference=str(REF),visual_QA=str(AOUT),spatial_results=str(RUNNEW),production=str(PROD),structure_status=STRUCT_STATUS,nicfi_status=NIC_STATUS,nicfi_results=str(NIC_RUN),cleanup_authorized=False))
print('Within-landscape S2/embedding workflow complete. NICFI status:',NIC_STATUS)
